# 06 - The cDC2-pDC streak and the annotation of record

* 12 / 12b: the streak between cDC2 and pDCs on the strict TotalVI UMAP (latent from heavy notebook 12; record linked).
  Not doublets (UMIs lower, not higher; scDblFinder score flat; Klk1 absent; Ly6c2 peaks in the middle): transitional DCs.
* 13a: evidence per strict Leiden r3 cluster (Table S1 scores, classic markers) + DC-block subclusters.
* 13b: the annotation of record (overrides listed in the 13b docstring). Result of record: 16 of 17 Fig 1B types (no HsPC
  cluster) + Transitional DC (735) + Mesothelial (359); 6,859 unresolved (kept, excluded from SAMap).

The DC subclusters come from a GPU Leiden; 13b names subcluster 3, so the record's subclusters are restored if they differ.

**Runtime:** ~25 min
**Envs used (each step runs as a subprocess in its own env):** masld_integration (12, 12b), rapids-sc (13a, 13b)
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
link("guilliams2022/10_strict", "guilliams2022/11_totalvi_strict")   # strict TotalVI latent from the record unless rebuilt
fresh("guilliams2022/11_totalvi_strict/dc_streak")

### 12 - zoom on the DC region

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/12_dc_streak_check.py
"""Step 12 - check the cDC2 <-> pDC streak on the strict TotalVI UMAP (user, 2026-11-12).
Zoom on the DC region: r3 clusters, labels, doublet score, sample/digestion/gate, markers for pDC, cDC2, cDC1,
transitional DC (tDC: Siglech+ Cx3cr1+ Zeb2+ Itgax+, Leylek 2019 / Sulczewski 2023), proliferation and doublet partners.
Output <OUT>/11_totalvi_strict/dc_streak/: umap_dc_zoom.png, r3_dc_clusters.tsv."""
import os
import anndata as ad, numpy as np, pandas as pd, scanpy as sc, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"
D = f"{OUT}/11_totalvi_strict/dc_streak"; os.makedirs(D, exist_ok=True)
E = ad.read_h5ad(f"{OUT}/11_totalvi_strict/embeddings.h5ad")
U = E.obsm["X_umap_totalvi"]; lab = E.obs.annotation_curated.astype(str)
dc = lab.isin(["pDCs", "cDC2", "cDC1", "Mig. cDCs"]).values
lo, hi = np.percentile(U[dc], 0.5, 0), np.percentile(U[dc], 99.5, 0)
pad = (hi - lo) * 0.08; lo, hi = lo - pad, hi + pad
box = ((U >= lo) & (U <= hi)).all(1)
print("cells in DC box", box.sum(), flush=True)
A = ad.read_h5ad(f"{OUT}/10_strict/guilliams2022_fig1b_strict.h5ad", backed="r")
assert (A.obs_names == E.obs_names).all()
m = A[np.flatnonzero(box)].to_memory(); s = ad.AnnData(X=m.X.copy(), obs=m.obs[[]].copy(), var=A.var.copy()); del m
s.obs = E.obs.loc[s.obs_names].copy(); s.obsm["X_umap"] = U[box]
sc.pp.normalize_total(s, target_sum=1e4); sc.pp.log1p(s)
s.var_names = s.var["symbol"].astype(str).values; s.var_names_make_unique()
G = {"pDC": ["Siglech", "Ccr9", "Bst2", "Ly6d", "Klk1", "Cox6a2", "Tcf4"],
     "cDC2": ["Cd209a", "Sirpa", "Itgam", "Clec4a4", "Mgl2"], "cDC1": ["Xcr1", "Clec9a"],
     "tDC/preDC": ["Cx3cr1", "Zeb2", "Ly6c2", "Itgax", "Lyz2", "Mki67"],
     "partner": ["Cd3e", "Cd79a", "Clec4f", "Alb", "Stab2", "Ptprc"]}
genes = [g for v in G.values() for g in v if g in s.var_names]
s.obs["log10_umis"] = np.log10(s.obs.umis_final.astype(float) + 1)
cols = ["annotation_curated", "leiden_clean_r3", "scDblFinder_score", "log10_umis", "digestion", "sort_gate", "sample", "bridge_type"] + genes
fig = sc.pl.umap(s, color=cols, ncols=6, s=4, show=False, return_fig=True, legend_fontsize=5, frameon=False, legend_loc="on data")
fig.savefig(f"{D}/umap_dc_zoom.png", dpi=150, bbox_inches="tight")
r3 = s.obs.leiden_clean_r3.astype(str)
X = pd.DataFrame(s[:, genes].X.toarray(), index=s.obs_names, columns=genes)
t = X.groupby(r3.values).mean()
t.insert(0, "n", r3.value_counts().reindex(t.index).values)
t.insert(1, "label", s.obs.groupby(r3.values).annotation_curated.agg(lambda x: x.astype(str).mode().iat[0]))
t.insert(2, "frac_label", s.obs.groupby(r3.values).annotation_curated.agg(lambda x: (x.astype(str) == x.astype(str).mode().iat[0]).mean()))
t.insert(3, "dbl_score", s.obs.groupby(r3.values).scDblFinder_score.median())
t.insert(4, "umis", s.obs.groupby(r3.values).umis_final.median())
t.insert(5, "exvivo", s.obs.groupby(r3.values).digestion.agg(lambda x: (x.astype(str) == "Ex Vivo").mean()))
t.insert(6, "top_sample", s.obs.groupby(r3.values)["sample"].agg(lambda x: f"{x.astype(str).mode().iat[0]} {(x.astype(str)==x.astype(str).mode().iat[0]).mean():.2f}"))
t.insert(7, "umap", [f"{a:.1f},{b:.1f}" for a, b in s.obs.groupby(r3.values).apply(lambda x: s.obsm["X_umap"][s.obs_names.get_indexer(x.index)].mean(0))])
t = t[t.n >= 100].sort_values("label")
t.to_csv(f"{D}/r3_dc_clusters.tsv", sep="\t", float_format="%.3g")
pd.set_option("display.width", 400)
print(t.to_string(float_format=lambda v: f"{v:.2f}"), flush=True)

In [ ]:
run("12_dc_streak_check.py", "masld_integration")

### 12b - profile along the streak

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/12b_dc_streak_profile.py
"""Step 12b - profile the cDC2 -> pDC streak along its axis (bins of the projection between the cDC2 and pDC UMAP centres).
Doublet test: doublets of pDC x cDC2 would carry ~2x UMIs, higher scDblFinder score and co-expression of both full
programmes; transitional DCs (tDC) carry a graded Siglech/Ly6c2 -> Cd209a/Itgax switch with Zeb2 and normal UMIs.
Output <OUT>/11_totalvi_strict/dc_streak/streak_profile.tsv, streak_cells.tsv.gz, streak_samples.tsv."""
import anndata as ad, numpy as np, pandas as pd, scanpy as sc
OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"; D = f"{OUT}/11_totalvi_strict/dc_streak"
E = ad.read_h5ad(f"{OUT}/11_totalvi_strict/embeddings.h5ad")
U = E.obsm["X_umap_totalvi"]; lab = E.obs.annotation_curated.astype(str).values
# centres = medians of the core clusters (pDC r3 30, cDC2 r3 2)
r3 = E.obs.leiden_clean_r3.astype(str).values
c0, c1 = np.median(U[r3 == "2"], 0), np.median(U[r3 == "30"], 0)
v = c1 - c0; L = np.linalg.norm(v); v = v / L
t = (U - c0) @ v / L; d = np.abs((U - c0) @ np.array([-v[1], v[0]]))
sel = (t > -0.1) & (t < 1.1) & (d < 0.6)
A = ad.read_h5ad(f"{OUT}/10_strict/guilliams2022_fig1b_strict.h5ad", backed="r")
m = A[np.flatnonzero(sel)].to_memory(); s = ad.AnnData(X=m.X.copy(), obs=E.obs.iloc[np.flatnonzero(sel)].copy(), var=A.var.copy()); del m
sc.pp.normalize_total(s, target_sum=1e4); sc.pp.log1p(s)
s.var_names = s.var.symbol.astype(str).values; s.var_names_make_unique()
g = ["Siglech", "Ly6c2", "Ccr9", "Klk1", "Tcf4", "Cd209a", "Itgax", "Sirpa", "Zeb2", "Cx3cr1", "Lyz2", "H2-Ab1", "Esam", "Clec4a4", "Mki67"]
X = pd.DataFrame(s[:, g].X.toarray(), index=s.obs_names, columns=g)
o = s.obs; o["t"] = t[sel]; o["bin"] = pd.cut(o.t, [-0.1, 0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 1.1])
o["co_pDC_cDC2"] = (X.Siglech > 1) & (X.Cd209a > 1)
grp = o.groupby("bin", observed=True)
P = pd.DataFrame({"n": grp.size(), "pDC_lab": grp.annotation_curated.agg(lambda x: (x.astype(str) == "pDCs").mean()),
                  "cDC2_lab": grp.annotation_curated.agg(lambda x: (x.astype(str) == "cDC2").mean()),
                  "umis_med": grp.umis_final.median(), "dbl_med": grp.scDblFinder_score.median(),
                  "dbl_called_raw": grp.scDblFinder_class.agg(lambda x: (x.astype(str) == "doublet").mean()),
                  "co_Siglech_Cd209a": grp.co_pDC_cDC2.mean(), "exvivo": grp.digestion.agg(lambda x: (x.astype(str) == "Ex Vivo").mean()),
                  "n_samples": grp["sample"].nunique()})
P = P.join(X.groupby(o.bin.values, observed=True).mean())
pd.set_option("display.width", 400)
print(P.to_string(float_format=lambda v: f"{v:.2f}"))
P.to_csv(f"{D}/streak_profile.tsv", sep="\t", float_format="%.3g")
mid = o.t.between(0.25, 0.85)
st = pd.crosstab(o.loc[mid, "sample"].astype(str), o.loc[mid, "sort_gate"].astype(str))
st["n"] = st.sum(1); st = st.sort_values("n", ascending=False)
print("\nstreak middle (t 0.25-0.85):", int(mid.sum()), "cells, samples:\n", st.head(15).to_string())
st.to_csv(f"{D}/streak_samples.tsv", sep="\t")
pd.concat([o[["sample", "sort_gate", "digestion", "annotation_curated", "leiden_clean_r3", "t", "bin", "umis_final", "scDblFinder_score"]].astype({"bin": str}), X], axis=1).to_csv(f"{D}/streak_cells.tsv.gz", sep="\t")

In [ ]:
run("12b_dc_streak_profile.py", "masld_integration")

In [ ]:
compare_tsv("guilliams2022/11_totalvi_strict/dc_streak/streak_profile.tsv")
show("guilliams2022/11_totalvi_strict/dc_streak/umap_dc_zoom.png")

In [ ]:
guard(20)
fresh("guilliams2022/13_annotation_of_record")

### 13a - evidence tables

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/13a_aor_scores.py
"""Step 13a - annotation of record, evidence tables (user go 2026-11-12 16:30 CET).
Input: 10_strict (234,552, X = clean counts) + 11_totalvi_strict embeddings (leiden_clean_r3, 85 clusters).
1. Table S1 (mmc1) top-50 mouse DEGs per Fig 1B type, cluster-level on r3, z within cell- / nucleus-majority clusters (as 06f).
2. Classic marker means per r3 cluster, curated-label mode, doublet score, UMIs, suspension, digestion, sort gate.
3. DC block (r3 clusters whose curated mode is pDCs or cDC2): Leiden r1 on the TotalVI latent of that subset, marker profile
   per subcluster, for the transitional DC label (Siglech+ Ly6c2+ Cd209a+, Klk1-; check in 12b).
Writes <OUT>/13_annotation_of_record/: r3_evidence.tsv, r3_tableS1_scores.tsv, dc_sub_evidence.tsv, dc_sub.tsv.gz,
lognorm cache not written (recomputed in 13b)."""
import os
import anndata as ad, numpy as np, pandas as pd, scanpy as sc, scipy.sparse as sp
OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"; D = f"{OUT}/13_annotation_of_record"; os.makedirs(f"{D}/figures", exist_ok=True)
S1 = "/mnt/masld_vault/geo/GSE192740/paper_supplement_cell2022/1-s2.0-S0092867421014811-mmc1.xlsx"
E = ad.read_h5ad(f"{OUT}/11_totalvi_strict/embeddings.h5ad")
a = ad.read_h5ad(f"{OUT}/10_strict/guilliams2022_fig1b_strict.h5ad")
assert (a.obs_names == E.obs_names).all()
sc.pp.normalize_total(a, target_sum=1e4); sc.pp.log1p(a)
sym = a.var.symbol.astype(str).values
cl = E.obs.leiden_clean_r3.astype(str).values; cats = sorted(set(cl), key=int)
codes = pd.Categorical(cl, categories=cats).codes
ind = sp.csr_matrix((np.ones(len(cl)), (codes, np.arange(len(cl)))), shape=(len(cats), len(cl))); ncl = np.asarray(ind.sum(1)).ravel()
def means(genes, X=a.X, ind=ind, ncl=ncl, idx=cats):
    cols = np.where(np.isin(sym, genes))[0]
    return pd.DataFrame(np.asarray((ind @ X[:, cols]).todense()) / ncl[:, None], index=idx, columns=sym[cols]).T.groupby(level=0).mean().T
x = pd.read_excel(S1, sheet_name=None); markers = {}
for k, v in x.items():
    if k.startswith("Mouse") and k.endswith("DEGs"):
        g = v.sort_values("score", ascending=False).iloc[:50, 0].astype(str)
        markers[k.replace("Mouse ", "").replace(" DEGs", "")] = [s for s in g if s in set(sym)]
G = means(sorted({s for v in markers.values() for s in v}))
o = E.obs
fnuc = pd.Series(o.suspension.astype(str).values == "nucleus").groupby(cl).mean().reindex(cats)
grp = np.where(fnuc.values > 0.5, "nucleus", "cell")
score = pd.DataFrame(index=cats, columns=list(markers), dtype=float)
for gname in ("cell", "nucleus"):
    sel = grp == gname; Gs = G.loc[sel]; Z = (Gs - Gs.mean()) / Gs.std().replace(0, np.nan)
    for t_, gl in markers.items():
        score.loc[Gs.index, t_] = Z[gl].mean(1)
score.to_csv(f"{D}/r3_tableS1_scores.tsv", sep="\t", float_format="%.3g")
srt = np.sort(score.values, 1)
CLASSIC = ["Alb", "Cyp2e1", "Cyp2f2", "Clec4f", "Vsig4", "Stab2", "Kdr", "Ptprc", "Cd3e", "Cd8a", "Nkg7", "Ncr1", "Gzma", "Cd79a",
           "Siglech", "Klk1", "Xcr1", "Cd209a", "Ccr7", "S100a8", "Ly6c2", "Lyz2", "Reln", "Dcn", "Krt19", "Mcpt8", "Kit", "Cd34",
           "Mki67", "Hba-a1"]
C = means(CLASSIC)[[g for g in CLASSIC if g in sym]]
gb = o.groupby(cl, observed=True)
ev = pd.DataFrame({"n": ncl.astype(int), "frac_nuc": fnuc.values, "S1_best": score.idxmax(1), "S1_score": srt[:, -1],
                   "S1_second": score.apply(lambda r: r.drop(r.idxmax()).idxmax(), axis=1), "margin": srt[:, -1] - srt[:, -2]}, index=cats)
ev["curated_mode"] = gb.annotation_curated.agg(lambda s: s.astype(str).mode().iat[0]).reindex(cats)
ev["curated_frac"] = gb.annotation_curated.agg(lambda s: (s.astype(str) == s.astype(str).mode().iat[0]).mean()).reindex(cats)
ev["flag"] = gb.annotation_flag.agg(lambda s: ",".join(sorted(set(s.astype(str)) - {"", "nan", "none"}))).reindex(cats)
ev["dbl"] = gb.scDblFinder_score.median().reindex(cats); ev["umis"] = gb.umis_final.median().reindex(cats)
ev["exvivo"] = gb.digestion.agg(lambda s: (s.astype(str) == "Ex Vivo").mean()).reindex(cats)
ev["n_samp"] = gb["sample"].nunique().reindex(cats)
ev["top_samp"] = gb["sample"].agg(lambda s: f"{s.astype(str).mode().iat[0]}:{(s.astype(str) == s.astype(str).mode().iat[0]).mean():.2f}").reindex(cats)
ev = ev.join(C.round(2))
ev.to_csv(f"{D}/r3_evidence.tsv", sep="\t", float_format="%.3g")
pd.set_option("display.width", 500); pd.set_option("display.max_columns", 80)
print(ev.drop(columns=CLASSIC, errors="ignore").sort_values(["S1_best", "n"]).to_string(float_format=lambda v: f"{v:.2f}"), flush=True)
print(C.round(1).loc[ev.sort_values(["S1_best", "n"]).index].assign(lab=ev.S1_best).to_string(), flush=True)
# DC block subclustering
import rapids_singlecell as rsc
dcl = ev.index[ev.curated_mode.isin(["pDCs", "cDC2"])]
m = np.isin(cl, dcl)
t = ad.AnnData(obs=o[m][["sample", "digestion", "annotation_curated"]].copy(), obsm={"Z": E.obsm["X_totalvi"][m].astype(np.float32)})
rsc.pp.neighbors(t, use_rep="Z"); rsc.tl.leiden(t, resolution=1.0, random_state=0, key_added="sub")
sub = t.obs["sub"].astype(str).values; scats = sorted(set(sub), key=int)
sind = sp.csr_matrix((np.ones(m.sum()), (pd.Categorical(sub, categories=scats).codes, np.arange(m.sum()))), shape=(len(scats), m.sum()))
sn = np.asarray(sind.sum(1)).ravel()
DCG = ["Siglech", "Ccr9", "Klk1", "Ly6d", "Tcf4", "Ly6c2", "Zeb2", "Cx3cr1", "Cd209a", "Itgax", "Sirpa", "Clec4a4", "Mgl2", "H2-Ab1",
       "Esam", "Lyz2", "Mki67", "Ccr2"]
X = a.X[np.flatnonzero(m)]
cols = np.where(np.isin(sym, DCG))[0]
SM = pd.DataFrame(np.asarray((sind @ X[:, cols]).todense()) / sn[:, None], index=scats, columns=sym[cols]).T.groupby(level=0).mean().T[[g for g in DCG if g in sym]]
og = t.obs.groupby(sub)
sev = pd.DataFrame({"n": sn.astype(int), "r3_mode": pd.Series(cl[m]).groupby(sub).agg(lambda s: s.mode().iat[0]).reindex(scats).values,
                    "curated_mode": og.annotation_curated.agg(lambda s: s.astype(str).mode().iat[0]).reindex(scats).values,
                    "exvivo": og.digestion.agg(lambda s: (s.astype(str) == "Ex Vivo").mean()).reindex(scats).values,
                    "n_samp": og["sample"].nunique().reindex(scats).values,
                    "umis": pd.Series(o.umis_final.values[m]).groupby(sub).median().reindex(scats).values,
                    "dbl": pd.Series(o.scDblFinder_score.values[m]).groupby(sub).median().reindex(scats).values}, index=scats).join(SM.round(2))
U = E.obsm["X_umap_totalvi"][m]
sev["umap"] = [f"{U[sub == s, 0].mean():.1f},{U[sub == s, 1].mean():.1f}" for s in scats]
sev.to_csv(f"{D}/dc_sub_evidence.tsv", sep="\t", float_format="%.3g")
print(sev.to_string(float_format=lambda v: f"{v:.2f}"), flush=True)
pd.DataFrame({"dc_sub": sub}, index=t.obs_names).to_csv(f"{D}/dc_sub.tsv.gz", sep="\t")
print("done", flush=True)

In [ ]:
run("13a_aor_scores.py", "rapids-sc", gpu=0, tail=10)

In [ ]:
import pandas as pd, shutil
a = pd.read_csv(rec("guilliams2022/13_annotation_of_record/dc_sub.tsv.gz"), sep="\t", index_col=0)
b = pd.read_csv(rep("guilliams2022/13_annotation_of_record/dc_sub.tsv.gz"), sep="\t", index_col=0)
same = float((a.dc_sub.astype(str) == b.dc_sub.reindex(a.index).astype(str)).mean()); print("DC subclusters identical to record:", same)
if same < 1:
    for f in ["dc_sub.tsv.gz", "dc_sub_evidence.tsv"]:
        shutil.copy2(rec(f"guilliams2022/13_annotation_of_record/{f}"), rep(f"guilliams2022/13_annotation_of_record/{f}"))
    print("record DC subclusters restored (13b names subcluster 3)")
compare_tsv("guilliams2022/13_annotation_of_record/r3_evidence.tsv")

### 13b - annotation of record

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/13b_aor_apply.py
"""Step 13b - annotation of record (AoR) for the strict object, decisions from 13a evidence (2026-11-12 CET).
Rule: label = Table S1 best type per Leiden r3 cluster (11_totalvi_strict, z within cell / nucleus clusters), except:
  - transitional DC: DC-block subcluster 3 (13a dc_sub; Siglech+ Ly6c2+ Cd209a+ Klk1- Ccr2+; profile 12b)
  - r3 12 (basophils + Siglech-low Ccr9+ pDC-like cells): per-cell curated label kept (pDCs / Basophil), flag mixed_r3_12
  - r3 80 (Dcn+ Krt19+ Cd34+): Mesothelial if Msln and Upk3b > 1 (not a Fig 1B type), else Fibroblast
  - unresolved (excluded from SAMap): r3 1 (hepatocyte+LSEC nuclei, scDblFinder 0.33), 61 (Alb+Stab2+Kdr, 442 UMI),
    84 (Alb+Dcn, 23 cells, one sample), 0 and 35 (low-UMI nuclei, weak Table S1 scores)
  - r3 29 -> cDC2 (S1 margin 0 vs Monocyte; H2-Ab1 4.8, Cx3cr1/Ccr2+) flag moDC_like; r3 65 -> ILC1 (S1); r3 82, 60 -> Hepatocyte
  - flags only: 39 KC_with_LSEC_signal, 63 B_with_myeloid_signal_high_UMI, 67 KC_with_RBC, 79 KC_low_UMI, 83 proliferating_single_sample
Writes <OUT>/13_annotation_of_record/: guilliams2022_fig1b_aor.h5ad (+ _cells, _nuclei; X clean counts, layers counts raw,
obsm X_totalvi + X_umap), cells_aor.tsv.gz, r3_labels_aor.tsv, composition_aor.tsv, figures/."""
import anndata as ad, numpy as np, pandas as pd, scanpy as sc, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
OUT = "/mnt/masld_vault/cross_species_repro/guilliams2022"; D = f"{OUT}/13_annotation_of_record"
E = ad.read_h5ad(f"{OUT}/11_totalvi_strict/embeddings.h5ad")
a = ad.read_h5ad(f"{OUT}/10_strict/guilliams2022_fig1b_strict.h5ad")
assert (a.obs_names == E.obs_names).all()
ev = pd.read_csv(f"{D}/r3_evidence.tsv", sep="\t", index_col=0); ev.index = ev.index.astype(str)
sub = pd.read_csv(f"{D}/dc_sub.tsv.gz", sep="\t", index_col=0).dc_sub.astype(str)
r3 = E.obs.leiden_clean_r3.astype(str)
n = a.copy(); sc.pp.normalize_total(n, target_sum=1e4); sc.pp.log1p(n)
n.var_names = n.var.symbol.astype(str).values; n.var_names_make_unique()
m80 = (r3 == "80").values
msl = {g: float(n[m80, g].X.mean()) for g in ["Msln", "Upk3b", "Lrrn4", "Dcn", "Krt19", "Cd34", "Pdgfra"]}
print("r3 80 means:", msl, flush=True)
lab = r3.map(ev.S1_best).astype(object)
flag = pd.Series("", index=r3.index, dtype=object)
lab[r3.isin(["29"])] = "cDC2"; flag[r3 == "29"] = "moDC_like"
lab[r3 == "65"] = "ILC1"; lab[r3.isin(["82", "60"])] = "Hepatocyte"
m12 = r3 == "12"; lab[m12] = E.obs.annotation_curated.astype(str)[m12]; flag[m12] = "mixed_r3_12"
lab[r3 == "80"] = "Mesothelial" if (msl["Msln"] > 1 and msl["Upk3b"] > 1) else "Fibroblast"
for c, why in {"1": "hep_LSEC_mixed_nuclei", "61": "hep_LSEC_mixed", "84": "hep_fibro_mixed", "0": "low_UMI_nuclei", "35": "low_UMI_nuclei"}.items():
    lab[r3 == c] = "unresolved"; flag[r3 == c] = why
for c, why in {"39": "KC_with_LSEC_signal", "63": "B_with_myeloid_signal_high_UMI", "67": "KC_with_RBC", "79": "KC_low_UMI",
               "83": "proliferating_single_sample"}.items():
    flag[r3 == c] = why
tdc = sub.index[sub == "3"]; lab[tdc] = "Transitional DC"; flag[tdc] = "tDC_Siglech+Ly6c2+Cd209a+Klk1-"
print(lab.value_counts().to_string(), flush=True)
ORDER = ["Hepatocyte", "Endothelial", "Fibroblast", "Mesothelial", "Cholangiocyte", "KC", "Monocyte", "Neutrophil", "Basophil",
         "cDC1", "cDC2", "Mig. cDCs", "Transitional DC", "pDCs", "B cell", "T cell", "NK cell", "ILC1", "HsPCs", "unresolved"]
assert set(lab) <= set(ORDER), set(lab) - set(ORDER)
cats = [c for c in ORDER if c in set(lab)]
a.obs["annotation_aor"] = pd.Categorical(lab.values, categories=cats); a.obs["aor_flag"] = flag.values
a.obs["leiden_clean_r3"] = r3.values; a.obs["dc_sub"] = sub.reindex(a.obs_names).fillna("").values
a.obsm["X_totalvi"] = E.obsm["X_totalvi"]; a.obsm["X_umap"] = E.obsm["X_umap_totalvi"]
a.obs[["sample", "suspension", "digestion", "sort_gate", "leiden_clean_r3", "dc_sub", "annotation_aor", "aor_flag"]].to_csv(f"{D}/cells_aor.tsv.gz", sep="\t")
pd.crosstab(r3, a.obs.annotation_aor.values).to_csv(f"{D}/r3_labels_aor.tsv", sep="\t")
comp = pd.crosstab(a.obs.annotation_aor, a.obs.suspension); comp["total"] = comp.sum(1); comp["pct"] = (100 * comp.total / comp.total.sum()).round(2)
comp.to_csv(f"{D}/composition_aor.tsv", sep="\t"); print(comp.to_string(), "\ntotal", int(comp.total.sum()), flush=True)
a.uns["note"] = ("ANNOTATION OF RECORD 2026-11-12: strict object (10_strict) + TotalVI 11_totalvi_strict; labels per 13b docstring "
                 "(Table S1 top-50 per r3 cluster + listed overrides). X = clean counts, layers counts = raw UMI.")
for k in a.obs.columns:
    if a.obs[k].dtype == object: a.obs[k] = a.obs[k].astype(str)
a.write_h5ad(f"{D}/guilliams2022_fig1b_aor.h5ad", compression="gzip"); print("written all", a.shape, flush=True)
# figures
pal = dict(zip(cats, sc.pl.palettes.default_20[:len(cats)])); pal["unresolved"] = "#d9d9d9"
n.obs["annotation_aor"] = a.obs.annotation_aor.values; n.obsm["X_umap"] = a.obsm["X_umap"]
fig, axs = plt.subplots(1, 2, figsize=(18, 8))
sc.pl.umap(n, color="annotation_aor", palette=pal, ax=axs[0], show=False, frameon=False, s=1.5, legend_loc="on data", legend_fontsize=7,
           title=f"Guilliams 2022 mouse atlas, rebuilt: {a.n_obs:,} cells + nuclei")
n.obs["suspension"] = a.obs.suspension.values
sc.pl.umap(n, color="suspension", ax=axs[1], show=False, frameon=False, s=1.5, title="suspension")
for ax in axs:
    for c in ax.collections: c.set_rasterized(True)
fig.savefig(f"{D}/figures/umap_aor.png", dpi=200, bbox_inches="tight"); fig.savefig(f"{D}/figures/umap_aor.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
S1 = "/mnt/masld_vault/geo/GSE192740/paper_supplement_cell2022/1-s2.0-S0092867421014811-mmc1.xlsx"
x = pd.read_excel(S1, sheet_name=None); dm = {}
for k, v in x.items():
    if k.startswith("Mouse") and k.endswith("DEGs"):
        t = k.replace("Mouse ", "").replace(" DEGs", "")
        if t in cats:
            dm[t] = [g for g in v.sort_values("score", ascending=False).iloc[:, 0].astype(str) if g in n.var_names][:3]
dm["Transitional DC"] = ["Ly6c2", "Cd209a", "Siglech"]; dm["Mesothelial"] = ["Msln", "Upk3b"]
dm = {k: dm[k] for k in cats if k in dm}
k = n[n.obs.annotation_aor != "unresolved"]
k.obs["annotation_aor"] = k.obs.annotation_aor.cat.remove_unused_categories()
dp = sc.pl.dotplot(k, dm, groupby="annotation_aor", standard_scale="var", show=False, return_fig=True)
dp.savefig(f"{D}/figures/dotplot_tableS1_aor.png", dpi=200, bbox_inches="tight"); dp.savefig(f"{D}/figures/dotplot_tableS1_aor.pdf", bbox_inches="tight")
del n, k
import rapids_singlecell as rsc
for susp, tag in (("cell", "cells"), ("nucleus", "nuclei")):
    s = a[a.obs.suspension.astype(str).values == susp].copy()
    t = ad.AnnData(obs=s.obs[[]].copy(), obsm={"Z": s.obsm["X_totalvi"].astype(np.float32)})
    rsc.pp.neighbors(t, use_rep="Z"); rsc.tl.umap(t, random_state=0)
    s.obsm["X_umap_all"] = s.obsm["X_umap"]; s.obsm["X_umap"] = np.asarray(t.obsm["X_umap"])
    if susp == "nucleus":
        for kk in ("adt", "adt_measured"): s.obsm.pop(kk, None)
    s.write_h5ad(f"{D}/guilliams2022_fig1b_aor_{tag}.h5ad", compression="gzip")
    fig, ax = plt.subplots(figsize=(9, 8))
    sc.pl.umap(s, color="annotation_aor", palette=pal, ax=ax, show=False, frameon=False, s=2, legend_loc="on data", legend_fontsize=7,
               title=f"{tag}: {s.n_obs:,}")
    for c in ax.collections: c.set_rasterized(True)
    fig.savefig(f"{D}/figures/umap_aor_{tag}.png", dpi=200, bbox_inches="tight"); plt.close(fig)
    print("written", tag, s.shape, flush=True)
print("done", flush=True)

In [ ]:
run("13b_aor_apply.py", "rapids-sc", gpu=0)

In [ ]:
compare_tsv("guilliams2022/13_annotation_of_record/composition_aor.tsv")
show("guilliams2022/13_annotation_of_record/figures/umap_aor.png")
show("guilliams2022/13_annotation_of_record/figures/dotplot_tableS1_aor.png")